In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l4.1 Residual stream

> 🎯 **Goal:** Understand why a transformer block *adds* to its input instead of
> replacing it, and why that single `+` is what makes deep networks trainable.

**Why this matters.** Everything else in this unit (pre-norm, the MLP, the full
block) hangs off this one idea. The residual stream is the backbone that carries
information from the embeddings at the bottom all the way to the logits at the top.
Get this and the rest of the architecture clicks into place.

**The intuition.** Picture a conveyor belt running through the whole model. A token's
vector rides along it. Each layer does not lift the vector off the belt and stamp a
new one in its place, it reads what is on the belt, scribbles a note (`delta`), and
drops that note *onto* the belt. The belt keeps everything that was already there
plus the new note. Gradients ride the same belt backwards during training, reaching
early layers without fading away.

**The mechanics.** A sublayer `f` (attention or an MLP) computes a change, and we
write `x + f(x)`. The term `x` is the **identity path** (sometimes called the skip
connection); `f(x)` is the **residual** or **delta**, the small correction the
sublayer contributes. Because the output is a sum, the derivative through the `+`
is just 1: the gradient flowing back splits, one copy goes into `f`, one copy
passes straight through unchanged. That unchanged copy is why **vanishing
gradients** (signals shrinking toward zero across many layers) stop being a
showstopper, and why we can stack dozens of blocks.

In [ ]:
from torch import nn

torch.manual_seed(0)
dim = 8
x = torch.randn(1, 4, dim)
sublayer = nn.Linear(dim, dim)
delta = sublayer(x)
residual_out = x + delta   # the residual connection
print("output = input + a learned delta")

**What just happened.** We built one sublayer (`nn.Linear`), ran the input through
it to get `delta`, and added it back: `residual_out = x + delta`. The print line
is the whole idea in one sentence: output is input plus a learned change. Each
block in the real model does exactly this, so information placed on the stream by
layer 1 can survive untouched all the way to the output if no later layer chooses
to overwrite it.

The assert is the proof: subtracting the original `x` back out of `residual_out`
recovers `delta` almost exactly (the `atol=1e-5` allows for tiny floating-point
rounding). The identity path really did pass `x` through unchanged.

⚠️ **Common pitfalls**
- Forgetting the `+` and writing `residual_out = sublayer(x)`. That is a plain
  stack of layers, not a residual network, and it trains far worse when deep.
- Assuming the residual stream has a fixed width per layer. It does not: every
  sublayer's output must be the *same* shape as its input so the add lines up.
- Reading `delta` as 'the answer'. It is only a correction; the running sum is
  what carries meaning.

🏋️ **Try it yourself.** Replace `residual_out = x + delta` with just `delta` (no
add), then stack the same sublayer ten times in a loop and print the norm of the
output each step. Watch how fast the signal can blow up or collapse without the
skip connection holding it steady. Then add the `x +` back and rerun.

In [ ]:
# (x + delta) - x recovers delta up to floating-point cancellation.
assert torch.allclose(residual_out - x, delta, atol=1e-5)